In [ ]:
"""
sandbox_time.ipynb

A sandbox to develop a time-resolved class.

Author: Stellina X. Ao
Created: 2026-07-07
Last Modified: 2026-07-07
Python Version: 3.11.14
"""


import scienceplots  # noqa: F401
import shutup
import matplotlib.pyplot as plt

%load_ext autoreload
%autoreload 2

# pretty plots
plt.style.use(["nature"])
plt.rcParams["figure.dpi"] = 200
%matplotlib widget
%config InlineBackend.print_figure_kwargs = {'bbox_inches':None}

# suppress warnings :-)
shutup.please()

In [ ]:
subj_id = "MR82"
sess_id = "20251027_152036"

## init

In [ ]:
from sg.models import make_tre, Encoder, StrategyEncoder

encoder = make_tre(Encoder)(
    subj_id,
    sess_id,
    norm=True,
    stepsize_s=0.25,
)

encoder_mb = make_tre(StrategyEncoder)(
    subj_id,
    sess_id,
    norm=True,
    stepsize_s=0.25,
    strategy_filter="mb",
)

encoder_mf = make_tre(StrategyEncoder)(
    subj_id,
    sess_id,
    norm=True,
    stepsize_s=0.25,
    strategy_filter="mf",
)

In [ ]:
encoder.verify()
encoder_mb.verify()
encoder_mf.verify()

In [ ]:
encoder.view_fits(model="baseline")

In [ ]:
encoder.view_fits()

In [ ]:
from squiggs.renderers import StrategyWeightPETHRenderer
from squiggs.neuron_viewer import NeuronViewer
from core.data import get_psths_cond

regr = "response"
reg = "DMS"

if regr == "rewarded":
    regressor = "rewarded_corr"
    mode = "rewarded"
elif regr == "response":
    regressor = "response_left"
    mode = "response"


r = StrategyWeightPETHRenderer(
    weights_mb=encoder_mb.encoder_weights[:, encoder.reg_idxs[reg], :],
    weights_mf=encoder_mf.encoder_weights[:, encoder.reg_idxs[reg], :],
    regressor=regressor,
    dm_idxs=encoder.dm_idxs,
    relim=False,
    peths_mb=get_psths_cond(encoder_mb.psths[reg], encoder_mb.trial_data, mode=mode),
    peths_mf=get_psths_cond(encoder_mf.psths[reg], encoder_mf.trial_data, mode=mode),
    pres=encoder.tpre,
    posts=encoder.tpost,
    binwidth_s=encoder.stepsize_s,
    tbin_centers=encoder.tbin_centers,
)

# confidence interval for beta weights - find examples where have confidence in differnece btween strategies
# conditional balancing
nv = NeuronViewer(num_units=encoder.psths[reg].shape[0], render_func=r)

## response only

In [ ]:
from sg.models import make_tre, Encoder

encoder_response = make_tre(Encoder)(
    subj_id,
    sess_id,
    tv_keys=["response"],
    norm=False,
    stepsize_s=0.1,
)

In [ ]:
encoder_response.view_weights()